# 과제: 사용자 정의 층과 손실 함수를 사용한 신경망 만들기


## **1. 데이터 준비**
- 데이터셋: `sklearn.datasets`에서 제공하는 `make_regression` 데이터를 사용합니다.
- 데이터 처리:
  1. **훈련 데이터(80%)** 와 **테스트 데이터(20%)** 로 분리하세요.
  2. 입력 데이터를 **표준화**하여 평균 0, 표준편차 1로 변환하세요.

In [3]:
import tensorflow as tf
import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## 데이터셋 생성
X, y = make_regression(n_samples=1000, n_features=10, noise=0.1, random_state=42)
y = y.reshape(-1, 1)  ## 타겟 데이터 형태 맞추기

# 데이터 분리 (80% 훈련, 20% 테스트)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 데이터 표준화
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_train = scaler_X.fit_transform(X_train)
X_test = scaler_X.transform(X_test)
y_train = scaler_y.fit_transform(y_train)
y_test = scaler_y.transform(y_test)

## **2. 사용자 정의 층**
- `MyDenseLayer`라는 사용자 정의 층을 만드세요.
- 요구사항:
  1. 입력 크기에 따라 **가중치(weight)** 와 **편향(bias)** 를 생성하고 학습 가능하도록 설정합니다.
  2. 입력 데이터를 받아 **ReLU 활성화 함수**를 적용합니다.

In [4]:
# 2. 사용자 정의 층
class MyDenseLayer(tf.keras.layers.Layer):
    def __init__(self, units, activation = None):
        super().__init__()
        self.units = units
        self.activation = tf.keras.activations.get(activation)

    def build(self, input_shape):
        self.weight = self.add_weight(name = 'weight', shape=[input_shape[-1], self.units], initializer="glorot_uniform")
        self.bias = self.add_weight(name = 'bias', shape = [self.units], initializer="zeros")

    def call(self, inputs):
        return tf.nn.relu(tf.matmul(inputs, self.weight) + self.bias)

## **3. 사용자 정의 손실 함수**
- **후버 손실(Huber Loss)** 를 구현하세요.
- 요구사항:
  1. 임계값(`delta`)은 1로 설정합니다.
  2. 작은 오차는 **제곱 손실**, 큰 오차는 **선형 손실**로 처리합니다.

In [5]:
# 3. 사용자 정의 손실 함수
def huber_loss(y_true, y_pred, delta=1.0):
    error = y_true - y_pred
    is_small_error = tf.abs(error) <= delta
    squared_loss = tf.square(error)/2
    linear_loss = delta * tf.abs(error) - tf.square(delta)/2
    return tf.where(is_small_error, squared_loss, linear_loss)

## **4. 모델 설계 및 훈련**
- 사용자 정의 층과 후버 손실 함수를 사용하여 신경망을 설계합니다.
- 요구사항:
  1. **구조:** 2개의 은닉층(각 32개의 뉴런)과 1개의 출력층.
  2. **Optimizer:** Adam.
  3. **평가지표:** MSE (Mean Squared Error).
  4. **훈련:** 10 epoch, batch size=32.

In [6]:
# 4. 모델 설계
class CustomModel(tf.keras.Model):
    def __init__(self):
        super(CustomModel, self).__init__()
        self.dense1 = MyDenseLayer(32)
        self.dense2 = MyDenseLayer(32)
        self.out = tf.keras.layers.Dense(1)  # 출력층

    def call(self, inputs):
        x = self.dense1(inputs)
        x = self.dense2(x)
        return self.out(x)

# 모델 생성 및 컴파일
model = CustomModel()
model.compile(optimizer='adam', loss=huber_loss, metrics=['mse'])

# 모델 훈련
history = model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=2)

Epoch 1/10
25/25 - 1s - 42ms/step - loss: 0.3933 - mse: 0.9160
Epoch 2/10
25/25 - 0s - 2ms/step - loss: 0.2438 - mse: 0.5237
Epoch 3/10
25/25 - 0s - 6ms/step - loss: 0.1180 - mse: 0.2406
Epoch 4/10
25/25 - 0s - 5ms/step - loss: 0.0526 - mse: 0.1052
Epoch 5/10
25/25 - 0s - 2ms/step - loss: 0.0367 - mse: 0.0735
Epoch 6/10
25/25 - 0s - 2ms/step - loss: 0.0274 - mse: 0.0547
Epoch 7/10
25/25 - 0s - 2ms/step - loss: 0.0215 - mse: 0.0430
Epoch 8/10
25/25 - 0s - 3ms/step - loss: 0.0171 - mse: 0.0341
Epoch 9/10
25/25 - 0s - 6ms/step - loss: 0.0139 - mse: 0.0279
Epoch 10/10
25/25 - 0s - 4ms/step - loss: 0.0117 - mse: 0.0234


## **5. 평가 및 예측**
- 요구사항:
  1. 테스트 데이터에서 **MSE**를 출력하세요.
  2. 테스트 데이터 중 첫 번째 샘플의 **예측값**과 **실제값**을 출력하세요.

In [8]:
# 5. 평가 및 예측
# 테스트 데이터 평가
test_loss, test_mse = model.evaluate(X_test, y_test, verbose=0)
print(f"테스트 데이터에서의 MSE: {test_mse}")

테스트 데이터에서의 MSE: 0.02908574417233467


In [10]:
# 첫 번째 샘플 예측값과 실제값 출력
y_pred = model.predict(X_test[:1])
y_pred = scaler_y.inverse_transform(y_pred)
y_actual = scaler_y.inverse_transform(y_test[:1])
print(f"첫 번째 샘플의 예측값: {y_pred[0][0]}, 실제값: {y_actual[0][0]}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
첫 번째 샘플의 예측값: 59.7575569152832, 실제값: 42.67137812993431
